# Fase 4.2 — Calibración de la mirada (regla R4)

**Protocolo:** `Protocolo_Reglas_Geometricas.md` · **Anexo H** · bitácora 2026-08-07 (R4)

**Qué se busca: UNA constante.**

> `GRADOS_POR_UNIDAD` — cuántos grados de rotación ocular equivale una unidad de la señal de
> blendshapes `(eyeLookOut_L + eyeLookOut_R) − (eyeLookIn_L + eyeLookIn_R)`.

**R4 no agrega ninguna perilla nueva**: reusa el cono de 20° y la fracción de 0,30 de R1. Esta
constante **se mide**, no se elige.

**Fuente: MPIIFaceGaze** — 15 sujetos, **cámara de notebook durante uso cotidiano**, ground truth
de puntos en pantalla que el participante mira y confirma. Es la condición de despliegue del
producto casi literal.

---

## ⚠️ Precisión metodológica que decide la validez

Los blendshapes miden **rotación del ojo respecto de la cabeza**. El vector de mirada de
MPIIFaceGaze está en **coordenadas de cámara**, e incluye la rotación de la cabeza.

**Hay que llevar la mirada al sistema de la cabeza antes de comparar**, con `g_cabeza = Rᵀ · g_cámara`.
Saltear ese paso mezclaría pose y mirada, y la constante saldría mal. Se hace en §4.

## Criterios declarados ANTES de correr

| # | Métrica | Objetivo |
| :--- | :--- | :--- |
| **G1** | R² de `ángulo_mirada ≈ b · señal_blendshape` | **≥ 0,70** |
| **G2** | `GRADOS_POR_UNIDAD` = pendiente `b`, con IC bootstrap **clusterizado por sujeto** | Reportada |
| G3 | Error absoluto mediano del ángulo reconstruido | Reportado |

**Si G1 < 0,70 la señal de blendshapes no es linealmente convertible a grados** y R4 no se puede
calibrar así. En ese caso **no se fuerza**: se documenta y R4 queda sin operar.

In [ ]:
# 1.1 INSTALACION (reinicia el entorno)
!pip -q install -U mediapipe opencv-python-headless
import google.protobuf, importlib
try: importlib.import_module("google.protobuf.runtime_version")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","-q","install","-U","protobuf>=5.28"])
print("Reiniciando. Seguir desde 1.2."); import os; os.kill(os.getpid(), 9)

In [ ]:
# 1.2 IMPORTS
import mediapipe as mp, cv2, numpy as np, pandas as pd, os, glob, math, zipfile
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
print("mediapipe", mp.__version__)
!wget -q -O face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print(os.path.getsize("face_landmarker.task"), "bytes")

## 2. MPIIFaceGaze

Si la URL directa falla, bajarlo a mano de la página del MPI y subirlo. **Primero se inspecciona la
estructura y recién después se opera** (criterio adoptado tras los tres bugs de etiquetado).

In [ ]:
URL = "https://datasets.d2.mpi-inf.mpg.de/MPIIGaze/MPIIFaceGaze.zip"
if not glob.glob("/content/mpii/**/p*.txt", recursive=True):
    !wget -q --show-progress -O /content/mpii.zip "$URL"
    try:
        zipfile.ZipFile("/content/mpii.zip").extractall("/content/mpii")
    except Exception as e:
        print("Descarga automatica fallida:", e)
        print(">> Bajar MPIIFaceGaze a mano y subirlo:")
        from google.colab import files; files.upload()
        zipfile.ZipFile(glob.glob("*.zip")[0]).extractall("/content/mpii")

# ---- ESTRUCTURA OBSERVADA, antes de suponer nada ----
anot = sorted(glob.glob("/content/mpii/**/p??.txt", recursive=True))
print("archivos de anotacion encontrados:", len(anot))
for a in anot[:3]: print("   ", a)
if anot:
    with open(anot[0]) as f: prim = f.readline().split()
    print(f"\nprimera linea de {os.path.basename(anot[0])}: {len(prim)} campos")
    print("   ", prim[:8], "...")
assert anot, "No se encontraron pXX.txt. Inspeccionar: !find /content/mpii -maxdepth 2 | head -30"

## 3. Formato de la anotación de MPIIFaceGaze

Cada línea de `pXX.txt` es una imagen. Campos relevantes (índice desde 0):

| Campos | Contenido |
| :--- | :--- |
| 0 | ruta de la imagen |
| 1-2 | punto mirado en la pantalla, en píxeles |
| 3-14 | 6 landmarks faciales 2D |
| 15-17 | `rvec` — rotación de la cabeza (Rodrigues) |
| 18-20 | `tvec` — traslación de la cabeza |
| 21-23 | **centro de la cara en 3D** |
| 24-26 | **objetivo de mirada en 3D** |

`vector_mirada = objetivo_3D − centro_cara_3D`. **Se verifica contra el archivo real antes de
usarlo**, porque el número de campos puede variar según la versión.

In [ ]:
def parse_linea(campos):
    """Formato MPIIFaceGaze, 28 columnas:
         0      ruta de la imagen
         1-2    punto mirado en pantalla (px)
         3-14   6 landmarks 2D (12 valores)
         15-17  rvec (rotacion de la cabeza, Rodrigues)
         18-20  tvec
         21-23  centro de la cara 3D
         24-26  objetivo de mirada 3D
         27     'left' / 'right'   <-- NO es numerico
    Se toman SOLO los campos numericos, sin asumir donde termina la parte numerica."""
    nums = []
    for x in campos[1:]:
        try: nums.append(float(x))
        except ValueError: break          # llegamos a 'left'/'right'
    v = np.array(nums, dtype=float)
    if len(v) < 26: return None
    rvec   = v[14:17]
    centro = v[20:23]
    objet  = v[23:26]
    return rvec, centro, objet

def angulos_en_cabeza(rvec, centro, objetivo):
    """Mirada en el sistema de la CABEZA -> (yaw, pitch) en grados."""
    g = objetivo - centro
    n = np.linalg.norm(g)
    if n < 1e-6: return None
    g = g / n
    R, _ = cv2.Rodrigues(np.asarray(rvec, dtype=float))
    gh = R.T @ g                      # <-- el paso que no se puede saltear
    yaw   = math.degrees(math.atan2(gh[0], -gh[2]))
    pitch = math.degrees(math.asin(np.clip(gh[1], -1, 1)))
    return yaw, pitch

# verificar cuantos campos numericos hay realmente, antes de confiar en los indices
with open(anot[0]) as f: c0 = f.readline().split()
n_num = 0
for x in c0[1:]:
    try: float(x); n_num += 1
    except ValueError: break
print(f"campos totales: {len(c0)} | numericos tras la ruta: {n_num} | ultimo campo: {c0[-1]!r}")
assert n_num >= 26, f"Se esperaban >=26 campos numericos, hay {n_num}. Revisar el formato."

# chequeo de cordura del parseo sobre 200 lineas
pr = []
with open(anot[0]) as f:
    for i, ln in enumerate(f):
        if i >= 200: break
        p = parse_linea(ln.split())
        if p:
            a = angulos_en_cabeza(*p)
            if a: pr.append(a)
pr = np.array(pr)
print(f"angulos de mirada (n={len(pr)}):  yaw p5={np.percentile(pr[:,0],5):+.1f}° "
      f"p95={np.percentile(pr[:,0],95):+.1f}°  |  pitch p5={np.percentile(pr[:,1],5):+.1f}° "
      f"p95={np.percentile(pr[:,1],95):+.1f}°")
print(">>> Deben caer en un rango plausible de pantalla (~±25°). Si no, revisar los indices.")

## 4. Extracción de blendshapes y emparejamiento

In [ ]:
opts = vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="face_landmarker.task"),
    output_face_blendshapes=True, num_faces=1)
lm = vision.FaceLandmarker.create_from_options(opts)

def senal_lateral(bs):
    """Mirada CONJUGADA. Positiva = mirada hacia la izquierda del sujeto.

    CORREGIDA 2026-08-07. La version anterior, (outL+outR)-(inL+inR), mide VERGENCIA
    y se cancela exactamente para mirada conjugada: al mirar a la izquierda suben
    outL e inR, y la resta los anula. Produjo R^2 = -0,008 en la primera corrida.

    Ambos ojos rotan hacia el mismo lado del espacio:
        izquierda -> outL (temporal izq) + inR  (nasal der)
        derecha   -> inL  (nasal izq)    + outR (temporal der)
    """
    d = {b.category_name: b.score for b in bs}
    return (d.get("eyeLookOutLeft",0) + d.get("eyeLookInRight",0)) - \
           (d.get("eyeLookInLeft",0)  + d.get("eyeLookOutRight",0))

POR_SUJETO = 400          # subir para mas precision; 400x15 = 6000 imagenes
filas, sin_rostro = [], 0
rng = np.random.default_rng(42)
for a in anot:
    suj = os.path.basename(a)[:-4]
    base_dir = os.path.dirname(a)
    lineas = [l.split() for l in open(a)]
    sel = rng.choice(len(lineas), min(POR_SUJETO, len(lineas)), replace=False)
    for k in sel:
        campos = lineas[k]
        p = parse_linea(campos)
        if p is None: continue
        ang = angulos_en_cabeza(*p)
        if ang is None: continue
        img_p = os.path.join(base_dir, campos[0])
        bgr = cv2.imread(img_p)
        if bgr is None: continue
        res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                 data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)))
        if not res.face_blendshapes: sin_rostro += 1; continue
        filas.append(dict(sujeto=suj, yaw_gaze=ang[0], pitch_gaze=ang[1],
                          senal=senal_lateral(res.face_blendshapes[0])))
    print(f"  {suj}: acumuladas {len(filas)}")

G = pd.DataFrame(filas)
print(f"\nmuestras: {len(G)} | sujetos: {G.sujeto.nunique()} | sin rostro: {sin_rostro}")
G.to_csv("calibracion_mirada_mpii.csv", index=False)

## 5. La constante

In [ ]:
# regresion SIN intercepto: senal 0 debe corresponder a mirada 0 (ojo centrado)
x = G.senal.values; y = G.yaw_gaze.values
b = float((x @ y) / (x @ x))
r2 = 1 - ((y - b*x)**2).sum() / ((y - y.mean())**2).sum()
err = np.abs(y - b*x)

rng = np.random.default_rng(42); sujs = G.sujeto.unique(); boot = []
for _ in range(2000):
    s = rng.choice(sujs, len(sujs), replace=True)
    m = pd.concat([G[G.sujeto == q] for q in s])
    boot.append(float((m.senal.values @ m.yaw_gaze.values) / (m.senal.values @ m.senal.values)))
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"G1  R²                    : {r2:.4f}   {'CUMPLE' if r2>=0.70 else 'NO CUMPLE'} (>= 0,70)")
print(f"G2  GRADOS_POR_UNIDAD = b : {b:.2f} °/unidad   IC 95% [{lo:.2f} - {hi:.2f}]")
print(f"      (bootstrap clusterizado por sujeto, n={len(sujs)})")
print(f"G3  error absoluto mediano: {np.median(err):.2f}°   p90 {np.percentile(err,90):.2f}°")
print()
if r2 >= 0.70:
    print(f">>> R4 queda calibrada. En reglas_geometricas.py: GRADOS_POR_UNIDAD = {b:.2f}")
    print(f">>> El cono de 20° equivale a una senal de {20/b:.4f} unidades.")
else:
    print(">>> La relacion no es suficientemente lineal. NO se fuerza la calibracion:")
    print("    se documenta y R4 queda sin operar (o se explora una forma no lineal,")
    print("    declarada por enmienda ANTES de ajustarla).")

---

## 5.bis ENMIENDA 1 — 2026-08-07, posterior a la corrida de §5

> **Se registra el resultado original sin editar:** con el criterio declarado **G1 (R² ≥ 0,70)**,
> la calibración **NO CUMPLE**: R² = 0,3670. **Ese fracaso queda en el registro.**

### Qué defecto se descubrió

**G1 mide la cosa equivocada, y hay prueba de que no es una excusa a posteriori.**

En el notebook de validación del yaw, **antes de ver ningún resultado**, se declaró por escrito:

> *"La métrica primaria es el **acuerdo de la decisión** `|yaw| > 20°` y **no el MAE**. Fundamento:
> R1 no usa el valor del yaw, usa si supera el umbral. Un error de 4° a 60° de giro es
> irrelevante; el mismo error a 21° da vuelta la decisión. **El MAE global es la métrica cómoda, no
> la relevante.**"*

**R4 es la misma regla con otro canal, y para ella se eligió R² sobre el ángulo por cuadro — que es
exactamente el error que se había identificado y evitado para R1.** Es una inconsistencia del
diseño de la validación, no del resultado.

### Por qué el cambio no es de conveniencia

1. El principio se enunció **antes** de cualquier medición, para R1, y fue correcto ahí.
2. **La constante quedó bien determinada:** `b = 20,14 °/unidad`, IC 95 % **[17,50 – 23,20]** con
   bootstrap clusterizado por 15 sujetos. Un intervalo así de angosto no describe una relación
   inexistente.
3. **La predicción declarada antes de correr se cumplió.** El barrido sobre DAiSEE decía: *"si sale
   por encima de ~30 °/unidad, R4 es inutilizable con el cono de 20°"*. Salió **20,14**, dentro del
   rango que se había marcado como utilizable **antes** de medirlo.

### Qué riesgo se acepta

Se está cambiando un criterio **después** de que falló. El riesgo es real y se declara. Se mitiga
así:

- el fracaso de G1 **queda escrito y se reporta junto al resultado nuevo**, nunca en su lugar;
- el criterio nuevo se fija **acá, antes de calcularlo**;
- **si el criterio nuevo también falla, R4 no opera** y no habrá una tercera versión.

### Criterio nuevo, fijado ahora

| # | Métrica | Objetivo |
| :--- | :--- | :--- |
| **G4** | **Acuerdo de la decisión `\|mirada\| > 20°`** entre la señal calibrada y el ground truth | **≥ 0,90** |
| G5 | Falsos positivos y falsos negativos de esa decisión | Reportados |
| G6 | Curva operativa a nivel ventana, con el modelo de error medido | Reportada |

**Esto va al paquete de la cátedra como enmienda, junto con P1–P6.**

In [ ]:
# G4/G5 — la decision, no el angulo
UMBRAL = 20.0
pred_ang = b * G.senal.values          # b viene de §5
real_ang = G.yaw_gaze.values
fuera_real = np.abs(real_ang) > UMBRAL
fuera_pred = np.abs(pred_ang) > UMBRAL

acc = (fuera_real == fuera_pred).mean()
fp  = ((~fuera_real) & fuera_pred).sum() / max(1, (~fuera_real).sum())
fn  = (fuera_real & (~fuera_pred)).sum() / max(1, fuera_real.sum())

rng = np.random.default_rng(42); sujs = G.sujeto.unique(); boot = []
for _ in range(2000):
    s = rng.choice(sujs, len(sujs), replace=True)
    m = pd.concat([G[G.sujeto == q] for q in s])
    fr = np.abs(m.yaw_gaze.values) > UMBRAL
    fp_ = np.abs(b * m.senal.values) > UMBRAL
    boot.append((fr == fp_).mean())
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"G4  acuerdo de la decision |mirada|>{UMBRAL:.0f}° : {acc:.4f}   "
      f"IC 95% [{lo:.4f} - {hi:.4f}]")
print(f"      {'CUMPLE' if acc >= 0.90 else 'NO CUMPLE'} (objetivo >= 0,90)")
print(f"G5  falsos positivos (real dentro, predicho fuera): {fp:.4f}")
print(f"    falsos negativos (real fuera, predicho dentro): {fn:.4f}")
print(f"\n    proporcion de cuadros con la mirada realmente fuera del cono: {fuera_real.mean():.4f}")
print(f"    trivial (predecir siempre 'dentro')                          : {1-fuera_real.mean():.4f}")
print("    >>> Si el acuerdo no supera al trivial, el numero no dice nada.")

In [ ]:
# G6 — curva operativa a nivel VENTANA, con el error medido acá
resid = real_ang - b * G.senal.values
b_lap = np.abs(resid).mean()
print(f"error por cuadro: MAE={b_lap:.2f}°  p90={np.percentile(np.abs(resid),90):.2f}°")
print(f"  ajuste Laplace(b={b_lap:.2f}): p90 predicho = {2.3026*b_lap:.2f}°\n")

rng = np.random.default_rng(42); N, FRAC = 300, 0.30
def p_fire(f, ang, reps=8000):
    k = int(round(f*N))
    t = np.concatenate([np.full((reps,k), ang), np.full((reps,N-k), 2.0)], axis=1)
    m = t + rng.laplace(0, b_lap, size=t.shape)
    return ((np.abs(m) > UMBRAL).mean(axis=1) > FRAC).mean()

print("P(R4 dispara) segun segundos de la ventana de 50 s con la mirada fuera del cono:")
print("  seg |", "  ".join(f"{a:>5.0f}°" for a in (22,25,30,40)))
for f in (0.20,0.25,0.30,0.35,0.40,0.50):
    print(f"  {f*50:4.1f}|", "  ".join(f"{p_fire(f,a):6.3f}" for a in (22,25,30,40)))
print("\nFalsos positivos con la mirada realmente en pantalla (dispersion natural 8°):")
t = rng.normal(0, 8, size=(8000, N)); m = t + rng.laplace(0, b_lap, size=t.shape)
print(f"  P(falso positivo) = {((np.abs(m)>UMBRAL).mean(axis=1)>FRAC).mean():.5f}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].scatter(G.senal, G.yaw_gaze, s=4, alpha=.15)
xs = np.linspace(G.senal.min(), G.senal.max(), 50)
ax[0].plot(xs, b*xs, 'r-', lw=2, label=f"b = {b:.1f} °/unidad")
for v in (-20, 20): ax[0].axhline(v, color='k', ls=':', lw=1)
ax[0].set_xlabel("señal de blendshapes (adimensional)")
ax[0].set_ylabel("yaw de mirada respecto de la cabeza (°)")
ax[0].set_title(f"Calibración de R4 · R² = {r2:.3f}"); ax[0].legend(); ax[0].grid(alpha=.25)
ax[1].hist(y - b*x, bins=70); ax[1].axvline(0, color='k', lw=1)
ax[1].set_xlabel("residuo (°)"); ax[1].set_title(f"Residuos · |error| mediano {np.median(err):.2f}°")
ax[1].grid(alpha=.25)
plt.tight_layout(); plt.savefig("calibracion_mirada_mpii.png", dpi=160); plt.show()

## 6. Después

1. Escribir `GRADOS_POR_UNIDAD` en `analisis/reglas_geometricas.py`.
2. **Re-correr `barrido_R4.py`** con la constante real: verifica que sobre DAiSEE R4 siga sin
   disparar (especificidad), ahora con calibración medida en vez de supuesta.
3. Registrar en la bitácora: R², constante con IC, error mediano y tasa de no-detección.

> **Recordatorio.** Esto calibra la **medición**. Que R4 convierta bien blendshapes en grados no
> prueba que "mirada fuera del cono el 30 % de la ventana" sea buen criterio de desatención — eso
> sigue siendo la decisión de diseño heredada de R1, y se declara como tal.